# Essay Topic Modeling with BERTopic

> **Data sensitivity:** This notebook uses restricted admissions data. Run locally only — do not commit data files or outputs to the repository. Ensure `essay_assessment/data/` is listed in `.gitignore`.

**Expected CSV schema** (`essay_assessment/data/essays.csv`):
| column | type | notes |
|---|---|---|
| `student_id` | int/str | unique applicant identifier |
| `essay` | str | full essay text |
| `score` | float | reviewer score |

**Workflow:**
1. Load essay data from CSV
2. Preprocess text
3. Configure BERTopic components (tuned for ~15k docs)
4. Fit the topic model
5. Inspect topics and score distributions per topic
6. Export outputs


## Install dependencies (if needed)

In [ ]:

# Run this only once if packages are missing
!pip install bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas openai


## Imports

In [ ]:
import re
from pathlib import Path

import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

In [ ]:

# ── Dataset selection — change this one variable to switch corpora ─────────────
# 'real'      → real student essays (data/Essays_df.xlsx)
# 'synthetic' → ../synthetic_data/synthetic_essays.csv
# 'freeform'  → ../synthetic_data/freeform_essays.csv
DATASET = 'freeform'

_ds = DATASET
DATA_SOURCES = {
    'real':      Path('../data/real/essays.xlsx'),
    'synthetic': Path('../data/synthetic/synthetic_essays.csv'),
    'freeform':  Path('../data/freeform/freeform_essays.csv'),
}
CSV_PATH   = DATA_SOURCES[_ds]
OUTPUT_DIR = Path(f'../outputs/topics/{_ds}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_N     = None   # set an int to use a random subset during development
RANDOM_STATE = 42

print(f"Dataset : {DATASET}")
print(f"  Input  : {CSV_PATH}")
print(f"  Outputs: {OUTPUT_DIR}/")


## Step 1: Load essays

In [ ]:
def load_essays(csv_path=CSV_PATH, sample_n=SAMPLE_N):
    p = Path(csv_path)
    if p.exists():
        if p.suffix.lower() in ('.xlsx', '.xls'):
            df = pd.read_excel(p)
            df["essay"] = df["Essay Response"]
            df["score"] = df["Essay Score"]
            df = df.dropna(subset="Essay Score")
        else:
            df = pd.read_csv(p)
            # synthetic / freeform CSVs already have 'essay'; score may be absent
            if 'essay' not in df.columns:
                raise ValueError(f"CSV {p.name} has no 'essay' column. Columns: {list(df.columns)}")
            if 'score' not in df.columns:
                if 'proxy_score' in df.columns:
                    df['score'] = df['proxy_score']
                    print("No 'score' column — using 'proxy_score' as stand-in.")
                else:
                    df['score'] = float('nan')
        df = df.dropna(subset=['essay'])
        if 'student_id' not in df.columns:
            df["student_id"] = df.index
        missing = [c for c in ('essay', 'score') if c not in df.columns]
        if missing:
            raise ValueError(f'Missing columns: {missing}')
        print(f'Loaded {len(df):,} essays from {p}')
    else:
        print('CSV not found — using dummy dataset.')
        df = pd.DataFrame({
            'student_id': range(1, 9),
            'score':      [3.5, 4.0, 3.0, 4.5, 3.5, 4.0, 3.0, 4.5],
            'essay': [
                'Growing up in a small town, I never imagined standing before a crowd. '
                'I struggled with anxiety for years. But joining debate changed me. '
                'I learned my voice matters. Now I want to study psychology to help others find theirs.',
                'My mother works two jobs. Every morning I watch her leave before sunrise. '
                'That image drives everything I do. I maintain a 4.0 GPA not for myself, but for her. '
                'I intend to become an engineer and give her the life she deserves.',
                'I have always believed science can solve humanity\'s greatest problems. '
                'Studies show renewable energy can reduce emissions by up to 70%. '
                'I conducted my own solar panel efficiency research. '
                'The data confirmed innovation, not incremental change, is required.',
                'Some say art is a luxury. I disagree. After Hurricane Maria devastated my community, '
                'art became our therapy. I organized murals, open mics, poetry slams. '
                'Counter to those who dismiss the humanities, I argue they are essential infrastructure.',
                'I want to be a doctor. I shadowed a cardiologist for 80 hours last summer. '
                'I observed three open-heart surgeries. Every case reinforced my commitment. '
                'Medicine is not just a career — it is a calling I cannot ignore.',
                'Leadership is not about authority; it is about service. '
                'As student council president I launched a mental health awareness week. '
                'Attendance exceeded 600 students. I am proud of that impact.',
                'When I was eight my grandmother taught me to cook. '
                'Those afternoons shaped my identity more than any classroom. '
                'Food is culture, memory, and love. I plan to study nutrition science through that lens.',
                'The first time I coded a working program, I felt a joy I had never experienced. '
                'I built a tutoring app used by 300 students in my district. '
                'Technology, I realized, is the most scalable form of empathy.',
            ],
        })

    if sample_n and len(df) > sample_n:
        df = df.sample(sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f'Sampled {sample_n} essays for development run.')

    return df

df = load_essays()
df[['student_id', 'score', 'essay']].head()


## Step 2: Basic preprocessing

In [ ]:
def basic_clean(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["essay_clean"] = df["essay"].map(basic_clean)
docs = df["essay_clean"].tolist()
df[["essay", "essay_clean"]].head()

## Step 3: Build BERTopic model

In [ ]:

# ── HDBSCAN tuning — min_cluster_size scales with corpus size ─────────────────
# Rule of thumb: ~1-2% of corpus size, with a floor of 5.
# Too small → over-splits into hyper-specific single-essay topics.
# Too large → under-splits, merging distinct themes.
#
#   real      ~15 000 docs → 50   (~0.3%)
#   synthetic   ~900 docs → 15   (~1.7%)
#   freeform    ~600 docs → 15   (~2.5%)
_CLUSTER_SIZE = {
    'real':      50,
    'synthetic': 15,
    'freeform':  10,
}
_MIN_SAMPLES = {
    'real':      10,
    'synthetic': 3,
    'freeform':  2,
}

# Parameters tuned for each corpus.
umap_model = UMAP(
    n_neighbors=15,        # higher = broader neighborhood, more stable clusters at scale
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=_CLUSTER_SIZE[DATASET],
    min_samples=_MIN_SAMPLES[DATASET],
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2 if DATASET != 'real' else 5,  # lower min_df for small corpora
)

topic_model = BERTopic(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True,
)
print(f"HDBSCAN min_cluster_size={_CLUSTER_SIZE[DATASET]}, min_samples={_MIN_SAMPLES[DATASET]} for '{DATASET}' corpus")
topic_model


## Step 4: Fit model and assign topics

In [ ]:
topics, probs = topic_model.fit_transform(docs)
df["topic"] = topics
df[["student_id", "topic", "score", "essay"]].head()


## Step 5: Inspect discovered topics

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info

In [ ]:
for topic_id in topic_info["Topic"].tolist():
    if topic_id == -1:
        continue
    print(f"Topic {topic_id}: {topic_model.get_topic(topic_id)}")

In [ ]:
repr_docs = topic_model.get_representative_docs()
for topic_id, documents in repr_docs.items():
    print(f"\nTopic {topic_id}:")
    for d in documents[:2]:
        print(" -", d)


## Step 5c: LLM topic labeling (local — no data leaves your machine)

BERTopic's default names are auto-generated from the top-4 keywords (e.g. `0_leadership_service_community_students`). We can replace those with readable, human-quality labels using a **locally-run LLM** — no API key, no data transmitted.

**How it works:**
- `update_topics()` re-runs only the representation step — no re-embedding, no re-clustering.
- For each topic, a prompt with the top keywords + a few representative essay excerpts is passed to the local model, which returns a short descriptive label.

**Two local options (pick one):**

| Option | How | Best for |
|---|---|---|
| **A — HuggingFace pipeline** (default) | Downloads model to disk, runs in-process via `transformers` | Already have GPU + `transformers` installed |
| **B — Ollama** | Runs a local LLM server; BERTopic talks to it via OpenAI-compatible endpoint | Prefer easy model management (Llama 3, Mistral, etc.) |

For option B: install Ollama from https://ollama.com, run `ollama pull llama3.2` in a terminal, then swap to the commented block in the code cell below.


In [ ]:

import gc
import torch
from transformers import AutoTokenizer, pipeline as hf_pipeline
from bertopic.representation import BaseRepresentation

# -- Free previous pipeline before reloading (prevents CUDA OOM on re-runs) --
# globals().pop() removes the variable binding by name; del _var would only
# delete the loop variable itself, not the object the name points to.
for _name in ('llm_representation', 'generator', 'tokenizer'):
    if _name in globals():
        globals().pop(_name)
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

# -- Model selection ----------------------------------------------------------
# Qwen3 models ARE instruct models — there is no separate "-Instruct" variant.
# Pass enable_thinking=False in apply_chat_template to suppress <think> blocks.
# HF_MODEL = "Qwen/Qwen3-0.6B"
HF_MODEL = "Qwen/Qwen3-4B"  # better quality if labels are poor

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
generator = hf_pipeline(
    "text-generation",
    model=HF_MODEL,
    tokenizer=tokenizer,
    device=0,                 # GPU 0; set device=-1 for CPU
    torch_dtype="float16",
    max_new_tokens=20,
    do_sample=False,
)


class ChatLLMLabeler(BaseRepresentation):
    """BERTopic-compatible representation model using a chat-format instruct LLM.

    Inherits BaseRepresentation so BERTopic accepts it as a valid type.
    Uses apply_chat_template so the model receives properly formatted input.
    enable_thinking=False suppresses Qwen3's <think>...</think> reasoning block.
    """

    def __init__(self, generator, tokenizer, nr_docs=5):
        self.generator = generator
        self.tokenizer = tokenizer
        self.nr_docs = nr_docs

    def extract_topics(self, topic_model, documents, c_tf_idf, topics):
        repr_docs_map = topic_model.representative_docs_
        result = {}
        non_outlier = [t for t in topics if t != -1]
        print(f"Labeling {len(non_outlier)} topics …")
        for topic_id, topic_words in topics.items():
            if topic_id == -1:
                result[topic_id] = [("Outliers", 1)]
                continue
            keywords = ", ".join(w for w, _ in topic_words[:10])
            excerpts = "\n\n".join(
                f"Excerpt {i+1}: {d[:400]}"
                for i, d in enumerate(repr_docs_map.get(topic_id, [])[:self.nr_docs])
            )
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are an expert analyst of college admissions essays. "
                        "A topic model has been run on a large corpus of essays and grouped them into clusters. "
                        "Each cluster represents a recurring theme or subject across MANY different applicants' essays — "
                        "not a single applicant. Your job is to name what that shared theme is.\n\n"
                        "You will receive:\n"
                        "  • Top keywords that statistically define the cluster\n"
                        "  • Short excerpts from a few representative essays in the cluster\n\n"
                        "Respond with ONLY a concise 3-6 word thematic label that captures what this group of essays "
                        "has in common (e.g. 'Overcoming adversity through sport', 'Family immigrant experience', "
                        "'STEM research and innovation'). No explanation, no punctuation at the end."
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Top keywords for this cluster: {keywords}\n\n"
                        f"Representative excerpts from different essays in this cluster:\n{excerpts}\n\n"
                        "What shared theme do these essays have in common? Respond with only the 3-6 word label."
                    ),
                },
            ]
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False,   # suppress <think> blocks in Qwen3
            )
            out = self.generator(prompt, return_full_text=False,
                                 max_new_tokens=20, do_sample=False)
            raw = out[0]["generated_text"].strip()
            # Take only the first non-empty line; strip stray punctuation
            label = next(
                (ln.strip(" .-_") for ln in raw.split("\n") if ln.strip(" .-_")),
                "Unlabeled",
            )
            print(f"  Topic {topic_id:3d}: {label}")
            result[topic_id] = [(label, 1)]
        return result


llm_representation = ChatLLMLabeler(generator, tokenizer, nr_docs=5)

# -- Option B: Ollama (local server, OpenAI-compatible endpoint) --------------
# 1. Install Ollama:  https://ollama.com
# 2. In a terminal:   ollama pull qwen3:0.6b
# 3. Uncomment the block below and comment out the Option A block above.
#
# import openai
# from bertopic.representation import OpenAI as BERTopicOpenAI
# client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# llm_representation = BERTopicOpenAI(
#     client, model="qwen3:0.6b", prompt=LABEL_PROMPT, chat=True,
#     nr_docs=5, delay_in_seconds=0,
# )

# -- Update topics in-place (no re-fitting needed) ----------------------------
topic_model.update_topics(docs, representation_model=llm_representation)

topic_info = topic_model.get_topic_info()
print(f"\nLabeled {len(topic_info) - 1} topics (excluding outlier cluster -1)")
topic_info[["Topic", "Count", "Name"]].head(20)


## Step 5b: Score distribution by topic

In [ ]:
import matplotlib.pyplot as plt

# Summary stats: mean score, essay count, and score std per topic
score_by_topic = (
    df[df["topic"] != -1]          # exclude outlier/noise cluster
    .groupby("topic")["score"]
    .agg(count="count", mean="mean", std="std")
    .sort_values("mean", ascending=False)
    .round(3)
)
score_by_topic = score_by_topic.join(
    topic_info.set_index("Topic")["Name"], how="left"
)
print(score_by_topic.to_string())


In [ ]:
# Box plot: score spread per topic (top 15 by essay count)
top_topics = score_by_topic.nlargest(15, "count").index.tolist()
plot_df = df[df["topic"].isin(top_topics)].copy()
plot_df["topic_label"] = plot_df["topic"].map(
    topic_info.set_index("Topic")["Name"].to_dict()
)

fig, ax = plt.subplots(figsize=(12, 6))
plot_df.boxplot(column="score", by="topic_label", ax=ax, rot=45)
ax.set_title("Score distribution by BERTopic topic (top 15 by size)")
ax.set_xlabel("Topic")
ax.set_ylabel("Score")
plt.suptitle("")
plt.tight_layout()
plt.show()


## Step 6: Optional topic reduction

In [ ]:
# Uncomment if you want fewer consolidated topics
#new_topics, new_probs = topic_model.reduce_topics(docs, nr_topics=50)

## Step 7: Save outputs

In [ ]:

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_DIR / 'essay_topics.csv', index=False)
topic_info.to_csv(OUTPUT_DIR / 'topic_info.csv', index=False)
topic_model.save(OUTPUT_DIR / 'bertopic_model', serialization='safetensors')

print('Saved outputs to:', OUTPUT_DIR)
